# Masked Aggregated Analysis — Federated AMR (6 Drugs)

Cross-drug comparison of federated learning results across all 4 DRIAMS sites, using **species-masked** runs.

**Data sources:**
- `08/Drugs/{drug}/{NN}-Run/results/final_results.csv` — latest run FL results
- `08/Drugs/{drug}/{NN}-Run/results/*_per_round.csv` — latest run per-round metrics
- `08/Drugs/{drug}/{NN}-Run/results/best_params_used.txt` — mask strategy used
- Set `TARGET_RUN` below to pin a specific run (None = latest)
- `07-01-results_dedicated_lr_mlp/` — Pooled (centralized) LR/MLP baselines
- `results_dedicated_lr_mlp/rf_results.pkl` — Pooled (centralized) RF baselines

**Selectable drugs**: configure `DRUGS_TO_ANALYZE` below. Results not yet available are silently skipped.

Compatible with Google Colab.

In [ ]:
!pip install seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, pickle, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
print("Aggregated analysis ready.")

In [ ]:
# ── CONFIGURE: mask strategy ──
MASK_STRATEGY = "majority"  # "none" | "union" | "majority" | "persite"

In [ ]:
if IN_COLAB:
    BASE = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed/Processing/Analysis")
else:
    BASE = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed/Processing/Analysis")
FL_DIR = BASE / "08-Federated-mlp-lr-rf" / "Drugs"
CSV_01_DIR = BASE / "07-Dedicated-MLP-Aggregated/07-01-results_dedicated_lr_mlp"
CSV_03_DIR = BASE / "07-Dedicated-MLP-Aggregated/results_dedicated_lr_mlp"

# ── CONFIGURE: target run ──
TARGET_RUN = None  # None = latest run; set to integer (e.g. 3) to pin a specific run

OUT_DIR = Path("./masked_aggregated_results"); OUT_DIR.mkdir(exist_ok=True)
print(f"FL dir: {FL_DIR}")
print(f"07 dir: {CSV_01_DIR}")

In [ ]:
# ── CONFIGURE: select which drugs to analyze ──
DRUGS_TO_ANALYZE = [
    "Ciprofloxacin",
    "Gentamicin",
    "Amoxicillin-Clavulanic_acid",
    "Piperacillin-Tazobactam",
    "Ceftriaxone",
    "Ceftazidime",
]
DRUG_NAME_MAP = {
    "Amoxicillin-Clavulanic_acid": "Amoxicillin-Clavulanic acid",
    "Piperacillin-Tazobactam": "Piperacillin-Tazobactam",
}

def parse_best_mask(results_dir):
    """Parse BEST_MASK from results/best_params_used.txt. Returns None if not found."""
    bp_file = results_dir / "best_params_used.txt"
    if not bp_file.exists():
        return None
    with open(bp_file, "r") as f:
        for line in f:
            m = re.match(r"BEST_MASK\s*=\s*(.+)", line.strip())
            if m:
                return m.group(1).strip().strip('"').strip("'")
    return None

def resolve_run_dir(drug):
    """Return (FL_DIR/drug/{NN}-Run/) for the requested or latest run."""
    runs = sorted([d for d in (FL_DIR / drug).glob("[0-9]*-Run") if d.is_dir()],
                  key=lambda p: int(p.name.split("-")[0]))
    if not runs:
        return None
    if TARGET_RUN is not None:
        match = [r for r in runs if r.name.startswith(f"{TARGET_RUN:02d}-")]
        return match[0] if match else None
    return runs[-1]

available = []
for drug in DRUGS_TO_ANALYZE:
    rd = resolve_run_dir(drug)
    if rd is None:
        print(f"  SKIP {drug}: no run directory found")
        continue
    results_dir = rd / "results"
    if not (results_dir / "final_results.csv").exists():
        print(f"  SKIP {drug}: no FL results yet")
        continue
    best_mask = parse_best_mask(results_dir)
    if best_mask != MASK_STRATEGY:
        print(f"  SKIP {drug}: mask '{best_mask}' != '{MASK_STRATEGY}' or best_params_used.txt missing")
        continue
    available.append(drug)

DRUGS = available
ANALYSIS_OK = len(DRUGS) > 0
print(f"\nAnalyzing {len(DRUGS)} drugs with MASK_STRATEGY='{MASK_STRATEGY}': {DRUGS}")
if not ANALYSIS_OK:
    print("\n*** No FL results found for this mask strategy. Run at least one federated.ipynb first. ***")

In [ ]:
# ── Load per-drug FL results ──
if not ANALYSIS_OK:
    print("Skipping data load — no drugs.")
    all_final, all_rounds, df_final = [], {}, pd.DataFrame()
else:
    all_final = []; all_rounds = {}
    SITE_ORDER = ["A","B","C","D"]
    for drug in DRUGS:
        rd = resolve_run_dir(drug)
        p = rd / "results" / "final_results.csv"
        df = pd.read_csv(p); df["Drug"] = drug; all_final.append(df)
        all_rounds[drug] = {}
        res_dir = rd / "results"
        for fname in ["fedavg_per_round.csv","fedlr_per_round.csv","fedrf_per_round.csv"]:
            rp = res_dir / fname
            if rp.exists():
                all_rounds[drug][fname.replace("_per_round.csv","")] = pd.read_csv(rp)
        for rp in sorted(res_dir.glob("fedprox_mu*_per_round.csv")):
            mu_str = rp.stem.replace("fedprox_mu","").replace("_per_round",""); mu = float(mu_str)
            all_rounds[drug][f"fedprox_mu{mu}"] = pd.read_csv(rp)
    df_final = pd.concat(all_final, ignore_index=True)
    print(f"Loaded {len(df_final)} rows ({len(DRUGS)} drugs)")
    for drug in DRUGS:
        n_m = df_final[df_final["Drug"]==drug]["Method"].nunique()
        n_r = sum(len(v) for v in all_rounds[drug].values()) if drug in all_rounds else 0
        print(f"  {drug}: {n_m} methods, {n_r} round-rows")

In [ ]:
# ── Load 07 pooled (centralized) baselines ──
if not ANALYSIS_OK:
    print("Skipping pooled — no FL data.")
else:
    pooled_ba = pd.read_csv(CSV_01_DIR / "results_balacc.csv")
    pooled_auc = pd.read_csv(CSV_01_DIR / "results_auc.csv")
    def csv_drug_name(d): return DRUG_NAME_MAP.get(d, d)
    rf_pkl = CSV_03_DIR / "rf_results.pkl"
    pooled_rf = {}
    if rf_pkl.exists():
        with open(rf_pkl, "rb") as f:
            pooled_rf = pickle.load(f)
    for _, row in pooled_ba.iterrows():
        drug_csv = row["Drug"]
        for dn in DRUGS:
            if csv_drug_name(dn) != drug_csv: continue
            for method, col in [("Pooled LR","LR"),("Pooled MLP","MLP")]:
                nr = {"Method":method,"Drug":dn}
                for s in SITE_ORDER:
                    nr[f"{s}_BalAcc"]=np.nan; nr[f"{s}_AUC"]=np.nan
                nr["All_BalAcc"]=row[col]
                auc_row = pooled_auc[pooled_auc["Drug"]==drug_csv]
                nr["All_AUC"] = auc_row[col].values[0] if len(auc_row)>0 else np.nan
                df_final = pd.concat([df_final, pd.DataFrame([nr])], ignore_index=True)
            if drug_csv in pooled_rf:
                test = pooled_rf[drug_csv].get("Test",{})
                nr = {"Method":"Pooled RF","Drug":dn}
                for s in SITE_ORDER:
                    nr[f"{s}_BalAcc"]=np.nan; nr[f"{s}_AUC"]=np.nan
                nr["All_BalAcc"]=test.get("BalAcc",np.nan)
                nr["All_AUC"]=test.get("AUC",np.nan)
                df_final = pd.concat([df_final, pd.DataFrame([nr])], ignore_index=True)
            break
    print(f"Total rows after pooling: {len(df_final)}")

In [ ]:
# ── Load unmasked (MASK_STRATEGY="none") runs for comparison ──
if not ANALYSIS_OK or len(df_final) == 0:
    print("Skipping unmasked load — no data.")
    df_final_unmasked = None
else:
    unmasked_all_final = []
    for drug in DRUGS:
        rd = resolve_run_dir(drug)
        if rd is None:
            continue
        results_dir = rd / "results"
        best_mask = parse_best_mask(results_dir)
        if best_mask == "none":
            p = results_dir / "final_results.csv"
            df = pd.read_csv(p); df["Drug"] = drug; unmasked_all_final.append(df)
    if unmasked_all_final:
        df_final_unmasked = pd.concat(unmasked_all_final, ignore_index=True)
        print(f"Loaded {len(df_final_unmasked)} unmasked rows ({len(unmasked_all_final)} drugs)")
    else:
        print("No unmasked (MASK_STRATEGY='none') runs found — comparison disabled.")
        df_final_unmasked = None

---
## Heatmaps: Per-Drug Balanced Accuracy + AUC

In [ ]:
# ── Heatmap grid ──
if not ANALYSIS_OK or len(df_final)==0:
    print("Skipping heatmaps — no data.")
else:
    fl_methods = [m for m in df_final["Method"].unique()
                  if "FL" in m or "FedAvg" in m or "FedProx" in m or "FedRF" in m
                  or "Cross-Site" in m or "Centralized" in m]
    fl_methods = [m for m in fl_methods if not m.startswith("Pooled")]
    keep_methods = fl_methods
    n_drugs = len(DRUGS); n_cols = min(3, n_drugs); n_rows = int(np.ceil(n_drugs / n_cols))
    cols_order = [f"Site {s}" for s in SITE_ORDER] + ["All"]

    def plot_heatmap_grid(metric, title, fname):
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5*n_cols, 4*n_rows), squeeze=False)
        for idx, drug in enumerate(DRUGS):
            ax = axes[idx//n_cols][idx%n_cols]
            sub = df_final[(df_final["Drug"]==drug) & (df_final["Method"].isin(keep_methods))]
            hm_data = {}
            for _, r in sub.iterrows():
                rd = {}
                for s in SITE_ORDER:
                    v = r.get(f"{s}_{metric}", np.nan)
                    rd[f"Site {s}"] = v if not pd.isna(v) else np.nan
                rd["All"] = r.get(f"All_{metric}", np.nan)
                hm_data[r["Method"]] = rd
            df_hm = pd.DataFrame(hm_data).T
            df_hm = df_hm[[c for c in cols_order if c in df_hm.columns and df_hm[c].notna().any()]]
            sns.heatmap(df_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.3, vmax=1.0,
                        linewidths=1.0, linecolor="white",
                        cbar_kws={"label": metric, "shrink":0.8}, ax=ax)
            ax.set_title(f"{drug}", fontsize=10, fontweight="bold")
            ax.set_xlabel(""); ax.set_ylabel("")
        for idx in range(n_drugs, n_rows*n_cols):
            axes[idx//n_cols][idx%n_cols].set_visible(False)
        fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.savefig(OUT_DIR / fname, bbox_inches="tight")
        plt.show()

    plot_heatmap_grid("BalAcc", "Federated AMR — Balanced Accuracy", "heatmap_grid_balacc.pdf")
    plot_heatmap_grid("AUC", "Federated AMR — AUC-ROC", "heatmap_grid_auc.pdf")

---
## Strategy Ranking

In [ ]:
# ── Strategy ranking ──
if not ANALYSIS_OK or len(df_final)==0:
    print("Skipping strategy ranking — no data.")
else:
    summary_all = df_final.groupby("Method")["All_BalAcc"].agg(["mean","std","count"]).reset_index()
    summary_all = summary_all[summary_all["count"] >= len(DRUGS)*0.5]
    summary_all = summary_all.sort_values("mean", ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(5, len(summary_all)*0.4)))
    colors_bar = []
    for m in summary_all["Method"]:
        if "FedAvg" in m and "LR" not in m and "RF" not in m: colors_bar.append("#ff7f0e")
        elif "FedProx" in m: colors_bar.append("#d62728")
        elif "LR" in m: colors_bar.append("#1f77b4")
        elif "RF" in m or "FedRF" in m: colors_bar.append("#2ca02c")
        elif "Centralized" in m: colors_bar.append("#333333")
        elif "Cross-Site" in m: colors_bar.append("#9467bd")
        else: colors_bar.append("#888888")
    ax.barh(summary_all["Method"], summary_all["mean"], xerr=summary_all["std"],
            color=colors_bar, edgecolor="white", capsize=3)
    ax.set_xlabel("Mean All-Site Balanced Accuracy (± std across drugs)")
    ax.set_title(f"Strategy Ranking ({len(DRUGS)} drugs)", fontsize=13, fontweight="bold")
    for i, (_, r) in enumerate(summary_all.iterrows()):
        ax.text(r["mean"]+r["std"]+0.005, i, f'{r["mean"]:.3f}±{r["std"]:.3f}', va="center", fontsize=8)
    ax.set_xlim(0, 1.05); ax.grid(True, ls="--", alpha=0.5, axis="x")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "strategy_ranking.pdf", bbox_inches="tight")
    plt.show()

---
## Convergence: Per-Drug Per-Round Progression

In [ ]:
# ── Convergence grid ──
if not ANALYSIS_OK or not all_rounds:
    print("Skipping convergence — no per-round data.")
else:
    n_drugs = len(DRUGS); n_cols_c = min(3, n_drugs); n_rows_c = int(np.ceil(n_drugs/n_cols_c))
    fig, axes = plt.subplots(n_rows_c, n_cols_c, figsize=(6*n_cols_c, 4*n_rows_c), squeeze=False)
    for idx, drug in enumerate(DRUGS):
        ax = axes[idx//n_cols_c][idx%n_cols_c]
        rounds_data = all_rounds.get(drug, {})
        colors = {"fedavg":"#ff7f0e","fedlr":"#1f77b4","fedrf":"#2ca02c"}
        if "fedprox" in method:
            colors[method] = "#d62728"  # all FedProx variants get red
        for method, df_round in rounds_data.items():
            if "All_BalAcc" not in df_round.columns: continue
            vals = df_round["All_BalAcc"].values
            c = colors.get(method,"#888888")
            ls = "-" if method=="fedavg" else "--" if "fedprox" in method else "-."
            lw = 2 if method == "fedavg" else 1.5
            label = method.replace("fedprox_mu","FedProx mu=").replace("fedavg","FedAvg MLP").replace("fedlr","FedAvg LR").replace("fedrf","FedRF")
            ax.plot(range(1,len(vals)+1), vals, color=c, ls=ls, lw=lw, label=label)
        sub = df_final[(df_final["Drug"]==drug) & (df_final["Method"].str.startswith("Pooled"))]
        for _, r in sub.iterrows():
            ax.axhline(r["All_BalAcc"], color="gray", ls=":", lw=1.5,
                       label=f'{r["Method"]} ({r["All_BalAcc"]:.3f})')
        ax.set_title(drug, fontsize=10, fontweight="bold")
        ax.set_xlabel("Round"); ax.set_ylabel("All-Site BalAcc")
        ax.legend(fontsize=6, loc="lower right"); ax.grid(True, ls="--", alpha=0.4)
        ax.set_ylim(0.3, 1.0)
    for idx in range(n_drugs, n_rows_c*n_cols_c):
        axes[idx//n_cols_c][idx%n_cols_c].set_visible(False)
    fig.suptitle("Per-Drug Convergence — All-Site Balanced Accuracy", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "convergence_grid.pdf", bbox_inches="tight")
    plt.show()

---
## Centralized vs Federated Gap

In [ ]:
# ── Centralized vs FL gap ──
if not ANALYSIS_OK or len(df_final)==0:
    print("Skipping gap analysis — no data.")
else:
    gap_rows = []
    for drug in DRUGS:
        sub = df_final[df_final["Drug"]==drug]
        pool_mlp = sub[sub["Method"]=="Pooled MLP"]
        if len(pool_mlp)==0: continue
        pool_ba = pool_mlp["All_BalAcc"].values[0]
        fl_sub = sub[(~sub["Method"].str.startswith("Pooled",na=False)) &
                     (~sub["Method"].str.startswith("Cross-Site",na=False))]
        if len(fl_sub)==0: continue
        best_fl = fl_sub.loc[fl_sub["All_BalAcc"].idxmax()]
        cs = sub[sub["Method"].str.startswith("Cross-Site",na=False)]
        cs_ba = cs["All_BalAcc"].max() if len(cs)>0 else np.nan
        gap_rows.append({"Drug":drug,"Pooled_MLP":pool_ba,"Best_FL_Method":best_fl["Method"],
                         "Best_FL_BA":best_fl["All_BalAcc"],"FL_Gap":pool_ba-best_fl["All_BalAcc"],
                         "Cross_Site_BA":cs_ba})
    if gap_rows:
        df_gap = pd.DataFrame(gap_rows).sort_values("FL_Gap",ascending=True)
        print(df_gap[["Drug","FL_Gap","Best_FL_Method"]].to_string(index=False))
        fig, ax = plt.subplots(figsize=(10,4))
        x = np.arange(len(df_gap)); w = 0.3
        ax.bar(x-w/2, df_gap["Pooled_MLP"], w, label="Pooled MLP", color="#333333")
        ax.bar(x+w/2, df_gap["Best_FL_BA"], w, label="Best FL", color="#ff7f0e")
        for i, r in df_gap.iterrows():
            ax.annotate(f"gap={r['FL_Gap']:.3f}", (i, r["Best_FL_BA"]+0.02), ha="center", fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels([d[:20] for d in df_gap["Drug"]], rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("All-Site Balanced Accuracy"); ax.set_title("Centralized vs Best Federated")
        ax.legend(fontsize=9); ax.set_ylim(0,1); ax.grid(True, ls="--", alpha=0.5, axis="y")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "centralized_vs_fl.pdf", bbox_inches="tight")
        plt.show()
    else:
        print("Not enough data for gap analysis.")

---
## Mask Delta Analysis

In [ ]:
# ── Mask Delta Analysis: masked vs unmasked ──
if not ANALYSIS_OK or len(df_final)==0 or df_final_unmasked is None or len(df_final_unmasked)==0:
    print("Skipping mask delta — no unmasked data for comparison.")
else:
    delta_rows = []
    for drug in DRUGS:
        sub_m = df_final[df_final["Drug"]==drug]
        sub_u = df_final_unmasked[df_final_unmasked["Drug"]==drug]
        if len(sub_m)==0 or len(sub_u)==0: continue
        for metric in ["BalAcc", "AUC"]:
            # best per-method for each drug (masked and unmasked)
            fl_m_sub = sub_m[(~sub_m["Method"].str.startswith("Pooled",na=False)) &
                             (~sub_m["Method"].str.startswith("Cross-Site",na=False))]
            fl_u_sub = sub_u[(~sub_u["Method"].str.startswith("Pooled",na=False)) &
                             (~sub_u["Method"].str.startswith("Cross-Site",na=False))]
            if len(fl_m_sub)==0 or len(fl_u_sub)==0: continue
            best_m = fl_m_sub.loc[fl_m_sub[f"All_{metric}"].idxmax()]
            best_u = fl_u_sub.loc[fl_u_sub[f"All_{metric}"].idxmax()]
            delta_rows.append({
                "Drug": drug,
                "Metric": metric,
                "Masked": best_m[f"All_{metric}"],
                "Unmasked": best_u[f"All_{metric}"],
                "Delta": best_m[f"All_{metric}"] - best_u[f"All_{metric}"]
            })
    if delta_rows:
        df_delta = pd.DataFrame(delta_rows)
        print(df_delta.to_string(index=False))
        fig, ax = plt.subplots(figsize=(10, 5))
        drugs_unique = df_delta["Drug"].unique()
        x = np.arange(len(drugs_unique))
        w = 0.35
        for i, metric in enumerate(["BalAcc", "AUC"]):
            sub = df_delta[df_delta["Metric"]==metric].set_index("Drug")
            vals = [sub.loc[d, "Delta"] if d in sub.index else 0 for d in drugs_unique]
            offset = (i - 0.5) * w
            bars = ax.bar(x + offset, vals, w, label=f"{metric} (masked - unmasked)",
                          color=["#1f77b4", "#2ca02c"][i])
            for j, v in enumerate(vals):
                if v != 0:
                    ax.text(x[j]+offset, v+0.002 if v>=0 else v-0.008,
                            f"{v:+.3f}", ha="center", fontsize=8, fontweight="bold")
        ax.axhline(0, color="black", lw=1)
        ax.set_xticks(x)
        ax.set_xticklabels([d[:20] for d in drugs_unique], rotation=45, ha="right", fontsize=9)
        ax.set_ylabel("Delta (Masked − Unmasked)")
        ax.set_title(f"Mask Delta Analysis — {MASK_STRATEGY} vs none", fontsize=13, fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, ls="--", alpha=0.5, axis="y")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "mask_delta.pdf", bbox_inches="tight")
        plt.show()
    else:
        print("Not enough data for mask delta analysis.")

---
## Master Summary Table

In [ ]:
# ── Summary table ──
if not ANALYSIS_OK or len(df_final)==0:
    print("Skipping summary — no data.")
else:
    pivot_ba = df_final.pivot_table(values="All_BalAcc", index="Drug", columns="Method", aggfunc="first")
    pivot_auc = df_final.pivot_table(values="All_AUC", index="Drug", columns="Method", aggfunc="first")
    print("Balanced Accuracy (All Sites):")
    print(pivot_ba.to_string(float_format=lambda x: f"{x:.3f}" if not pd.isna(x) else "   -"))
    print()
    print("AUC-ROC (All Sites):")
    print(pivot_auc.to_string(float_format=lambda x: f"{x:.3f}" if not pd.isna(x) else "   -"))
    pivot_ba.to_csv(OUT_DIR / "summary_balacc.csv"); pivot_auc.to_csv(OUT_DIR / "summary_auc.csv")

In [ ]:
print("\n" + "="*60)
print(f"  Masked aggregated analysis complete. {len(DRUGS)} drugs analyzed.")
print(f"  Results in {OUT_DIR.resolve()}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

---
**Done.**